# 08 — Freeze & QA Unified DEV Wave 1

**Version:** `SAJU_ML_V4_UNIFIED_DEV_WAVE1_FREEZE_QA_20260816`

This notebook freezes the event/source collection produced **after** the 100-subject roster freeze and **before any astrology scoring**.

It does not fit any astrology model and does not open any sealed holdout.

Expected Wave 1 situation:

- frozen roster: 100 subjects;
- some subjects may be `UNPAIRABLE_THIS_WAVE`;
- subject replacement is forbidden;
- same-year pairs are forbidden for the annual target;
- chronology is measured only after the wave is frozen;
- if fewer than 100 usable pairs remain, a fresh additional roster wave is required.

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib, json, math
import numpy as np
import pandas as pd

NOTEBOOK_VERSION = "SAJU_ML_V4_UNIFIED_DEV_WAVE1_FREEZE_QA_20260816"

def find_repo_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/"saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run from inside Chartpalja repo.")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

ROOT=find_repo_root()
ROSTER_DIR=ROOT/"research/ml/artifacts/v4_unified_dev_roster"
EVENT_DIR=ROOT/"research/ml_corpus/v4_unified_dev_events"
OUT=ROOT/"research/ml/artifacts/v4_unified_dev_wave1"
OUT.mkdir(parents=True,exist_ok=True)

paths={
 "roster":ROSTER_DIR/"V4_UNIFIED_DEV_SUBJECT_ROSTER_100.csv",
 "roster_freeze":ROSTER_DIR/"V4_UNIFIED_DEV_SUBJECT_ROSTER_FREEZE.json",
 "events":EVENT_DIR/"V4_UNIFIED_DEV_WAVE1_VERIFIED_EVENTS.csv",
 "pairs":EVENT_DIR/"V4_UNIFIED_DEV_WAVE1_SELECTED_PAIRS.csv",
 "unpairable":EVENT_DIR/"V4_UNIFIED_DEV_WAVE1_UNPAIRABLE_LEDGER.csv",
 "collection_summary":EVENT_DIR/"V4_UNIFIED_DEV_WAVE1_COLLECTION_SUMMARY.json",
}
missing=[k for k,p in paths.items() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing inputs: %s" % missing)

roster=pd.read_csv(paths["roster"])
events=pd.read_csv(paths["events"])
pairs=pd.read_csv(paths["pairs"])
unpairable=pd.read_csv(paths["unpairable"])

with open(paths["roster_freeze"],encoding="utf-8") as f:
    roster_freeze=json.load(f)
with open(paths["collection_summary"],encoding="utf-8") as f:
    collection_summary=json.load(f)

print("repo:",ROOT)
print("output:",OUT)

repo: /Users/sangjinlee/Desktop/projects/saju
output: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_unified_dev_wave1


## 1. Frozen-roster lineage and no-leakage checks

In [2]:
assert roster_freeze["status"]=="V4_UNIFIED_DEV_SUBJECT_UNIVERSE_FROZEN"
assert roster_freeze["n_subjects"]==100
assert roster_freeze["astrology_scored"] is False
assert roster_freeze["sealed_holdouts_loaded"] is False

assert len(roster)==100
assert roster.subject_id.nunique()==100
assert set(events.subject_id).issubset(set(roster.subject_id))
assert set(pairs.subject_id).issubset(set(roster.subject_id))
assert set(unpairable.subject_id).issubset(set(roster.subject_id))

pair_ids=set(pairs.subject_id)
unpair_ids=set(unpairable.subject_id)
assert pair_ids.isdisjoint(unpair_ids)
assert pair_ids | unpair_ids == set(roster.subject_id)

assert not pairs.astrology_scored.astype(bool).any()
assert not unpairable.astrology_scored.astype(bool).any()

print("Frozen membership: PASS")
print("No subject replacement: PASS")
print("No astrology scoring: PASS")
print("Sealed holdouts: NOT LOADED")

Frozen membership: PASS
No subject replacement: PASS
No astrology scoring: PASS
Sealed holdouts: NOT LOADED


## 2. Event/source/taxonomy integrity

In [3]:
required_event_cols={
 "subject_id","preassigned_axis","event_id","event_year","polarity",
 "event_type","description","source_1","source_2",
 "taxonomy_confirmed","polarity_confirmed","source_confirmed","annual_eligible"
}
assert required_event_cols.issubset(events.columns)

assert events.taxonomy_confirmed.astype(bool).all()
assert events.polarity_confirmed.astype(bool).all()
assert events.source_confirmed.astype(bool).all()
assert events.annual_eligible.astype(bool).all()
assert events.source_1.astype(str).str.startswith("http").all()
assert events.source_2.astype(str).str.startswith("http").all()
assert events.event_id.nunique()==len(events)

roster_axis=roster.set_index("subject_id")["preassigned_axis"]
assert all(events.set_index("subject_id")["preassigned_axis"] == roster_axis.loc[events.subject_id].values)

polarity_counts=events.groupby(["subject_id","polarity"]).size().unstack(fill_value=0)
assert (polarity_counts["positive"]>=1).all()
assert (polarity_counts["negative"]>=1).all()

print("verified event rows:",len(events))
print("event/source/taxonomy integrity: PASS")

verified event rows: 146
event/source/taxonomy integrity: PASS


## 3. Rebuild selected pairs deterministically from verified collected events

In [4]:
rebuilt=[]

for sid,g in events.groupby("subject_id"):
    pos=g[g.polarity=="positive"].copy()
    neg=g[g.polarity=="negative"].copy()
    candidates=[]
    for _,p in pos.iterrows():
        for _,n in neg.iterrows():
            gap=abs(int(p.event_year)-int(n.event_year))
            if gap==0:
                continue
            candidates.append({
                "positive_event_id":p.event_id,
                "negative_event_id":n.event_id,
                "positive_year":int(p.event_year),
                "negative_year":int(n.event_year),
                "year_gap":gap,
            })
    assert candidates, "Pairable subject has no distinct-year opposite-polarity candidate: %s"%sid
    c=pd.DataFrame(candidates).sort_values(
        ["year_gap","positive_year","negative_year","positive_event_id","negative_event_id"]
    ).iloc[0]
    rebuilt.append({"subject_id":sid,**c.to_dict()})

rebuilt=pd.DataFrame(rebuilt)

check=pairs.merge(rebuilt,on="subject_id",suffixes=("_provided","_rebuilt"))
assert len(check)==len(pairs)
assert (check.positive_year_provided==check.positive_year_rebuilt).all()
assert (check.negative_year_provided==check.negative_year_rebuilt).all()
assert (check.year_gap_provided==check.year_gap_rebuilt).all()

print("pair reconstruction from collected candidates: PASS")
print("NOTE: minimum-gap claim is limited to the bounded verified source sweep, not exhaustive whole-life events.")

pair reconstruction from collected candidates: PASS
NOTE: minimum-gap claim is limited to the bounded verified source sweep, not exhaustive whole-life events.


## 4. Wave 1 balance audit

In [5]:
pairs["positive_earlier_calc"]=pairs.positive_year < pairs.negative_year
pairs["abs_year_gap"]=(pairs.positive_year-pairs.negative_year).abs()

axis_summary=(
    pairs.groupby("axis")
    .agg(
        n_pairs=("subject_id","size"),
        n_subjects=("subject_id","nunique"),
        positive_earlier_share=("positive_earlier_calc","mean"),
        median_abs_year_gap=("abs_year_gap","median"),
    )
    .reset_index()
)

overall={
 "n_pairs":int(len(pairs)),
 "n_subjects":int(pairs.subject_id.nunique()),
 "positive_earlier_share":float(pairs.positive_earlier_calc.mean()),
 "median_abs_year_gap":float(pairs.abs_year_gap.median()),
 "same_year_pairs":int((pairs.abs_year_gap==0).sum()),
 "n_unpairable":int(len(unpairable)),
}

display(axis_summary)
print(json.dumps(overall,indent=2))

assert overall["same_year_pairs"]==0
chronology_pass=0.40 <= overall["positive_earlier_share"] <= 0.60

print("chronology 40-60 gate:",chronology_pass)

,axis,n_pairs,n_subjects,positive_earlier_share,median_abs_year_gap
0,COMPETITIVE,28,28,0.571429,1.0
1,PROJECT,19,19,0.052632,3.0
2,STATUS,26,26,0.730769,6.5


{
  "n_pairs": 73,
  "n_subjects": 73,
  "positive_earlier_share": 0.4931506849315068,
  "median_abs_year_gap": 3.0,
  "same_year_pairs": 0,
  "n_unpairable": 27
}
chronology 40-60 gate: True


## 5. Decide whether an additional unified DEV wave is required

In [6]:
TARGET_PAIRS={"COMPETITIVE":30,"PROJECT":25,"STATUS":45}
achieved={r.axis:int(r.n_pairs) for _,r in axis_summary.iterrows()}
deficits={axis:max(0,TARGET_PAIRS[axis]-achieved.get(axis,0)) for axis in TARGET_PAIRS}

# Wave-1 empirical pairability is used only for axis sample-size planning,
# never for chronology-direction selection.
roster_axis_counts=roster.preassigned_axis.value_counts().to_dict()
pairability={
 axis: achieved.get(axis,0)/float(roster_axis_counts[axis])
 for axis in TARGET_PAIRS
}

recommended_wave2={}
for axis in TARGET_PAIRS:
    deficit=deficits[axis]
    rate=pairability[axis]
    if deficit==0:
        recommended_wave2[axis]=0
    elif rate>0:
        recommended_wave2[axis]=int(math.ceil(deficit/rate))
    else:
        recommended_wave2[axis]=deficit*2

need_more=len(pairs)<100 or any(v>0 for v in deficits.values())

decision_status=(
 "V4_UNIFIED_DEV_WAVE1_FROZEN_NEEDS_ADDITIONAL_WAVE"
 if need_more else
 "V4_UNIFIED_DEV_READY_FOR_FEATURE_GENERATION"
)

decision={
 "version":"V4_UNIFIED_DEV_WAVE1_FREEZE_DECISION_V1",
 "notebook_version":NOTEBOOK_VERSION,
 "created_at":datetime.now().isoformat(timespec="seconds"),
 "status":decision_status,
 "wave1":{
   **overall,
   "axis_counts":achieved,
   "axis_pairability":pairability,
   "chronology_40_60":bool(chronology_pass),
   "collection_scope":"BOUNDED_MAJOR_EVENT_SOURCE_SWEEP",
 },
 "target_pairs":TARGET_PAIRS,
 "pair_deficits":deficits,
 "recommended_wave2_roster_axis_counts":recommended_wave2,
 "recommended_wave2_total_subjects":int(sum(recommended_wave2.values())),
 "rules":{
   "subject_replacement_wave1":False,
   "directional_chronology_backfill":False,
   "wave2_selection_may_use_axis_deficit":True,
   "wave2_selection_may_use_chronology_direction":False,
   "astrology_feature_generation_allowed":False if need_more else True,
 },
 "holdout_integrity":{
   "NEW_CONFIRM_loaded":False,
   "Validation_B_loaded":False,
   "Public_CHECK_loaded":False,
   "Public_FINAL_loaded":False,
 },
 "next_rule":(
   "Freeze Wave 1. If additional wave required, freeze a completely fresh subject roster "
   "using only axis deficits and observed pairability rates; do not condition membership on "
   "positive-earlier/later chronology and do not generate astrology features yet."
 ),
}

with open(OUT/"V4_UNIFIED_DEV_WAVE1_FREEZE_DECISION.json","w",encoding="utf-8") as f:
    json.dump(decision,f,ensure_ascii=False,indent=2)

axis_summary.to_csv(OUT/"V4_UNIFIED_DEV_WAVE1_AXIS_AUDIT.csv",index=False)
pairs.to_csv(OUT/"V4_UNIFIED_DEV_WAVE1_FROZEN_PAIRS.csv",index=False)

print(json.dumps(decision,ensure_ascii=False,indent=2))

{
  "version": "V4_UNIFIED_DEV_WAVE1_FREEZE_DECISION_V1",
  "notebook_version": "SAJU_ML_V4_UNIFIED_DEV_WAVE1_FREEZE_QA_20260816",
  "created_at": "2026-08-16T20:39:30",
  "status": "V4_UNIFIED_DEV_WAVE1_FROZEN_NEEDS_ADDITIONAL_WAVE",
  "wave1": {
    "n_pairs": 73,
    "n_subjects": 73,
    "positive_earlier_share": 0.4931506849315068,
    "median_abs_year_gap": 3.0,
    "same_year_pairs": 0,
    "n_unpairable": 27,
    "axis_counts": {
      "COMPETITIVE": 28,
      "PROJECT": 19,
      "STATUS": 26
    },
    "axis_pairability": {
      "COMPETITIVE": 0.9333333333333333,
      "PROJECT": 0.76,
      "STATUS": 0.5777777777777777
    },
    "chronology_40_60": true,
    "collection_scope": "BOUNDED_MAJOR_EVENT_SOURCE_SWEEP"
  },
  "target_pairs": {
    "COMPETITIVE": 30,
    "PROJECT": 25,
    "STATUS": 45
  },
  "pair_deficits": {
    "COMPETITIVE": 2,
    "PROJECT": 6,
    "STATUS": 19
  },
  "recommended_wave2_roster_axis_counts": {
    "COMPETITIVE": 3,
    "PROJECT": 8,
    "STAT

## What to send back

After **Kernel Restart → Run All**, send:

```text
research/ml/artifacts/v4_unified_dev_wave1/

V4_UNIFIED_DEV_WAVE1_FREEZE_DECISION.json
V4_UNIFIED_DEV_WAVE1_AXIS_AUDIT.csv
V4_UNIFIED_DEV_WAVE1_FROZEN_PAIRS.csv
```

Expected with this Wave 1 collection:

```text
V4_UNIFIED_DEV_WAVE1_FROZEN_NEEDS_ADDITIONAL_WAVE
```

Do **not** generate astrology features while that status is active.